# Verifying "[Working Note] What a Published Score Cannot Tell You"

Press **Run All**.  It takes seconds, needs no GPU and no internet.

`verify.py` performs 19 deterministic checks.  Each one PARSES a number out of the Working
Note and RECOMPUTES it from the committed data, then compares.  It fails if the note drifts
from the data or the data drifts from the note, which a checker that only re-ran the
producing scripts would miss.

Two checks read the competition SDK, which this notebook takes from the attached competition
rather than redistributing: the reachability proof, and the finding-7 comparison showing the
rail and the scorer omit `file` in the same way.

The note's SHA-256 is `3ef6503a643a608b`, printed back below so you can confirm the text checked
here is the text that was filed.


In [ ]:
import base64, io, zipfile, os, pathlib, hashlib
PAYLOAD = "..."
work = pathlib.Path('/kaggle/working/jed'); work.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(work)
note = (work / 'working_note_filed.md').read_bytes()
print('note chars:', len(note.decode()))
print('note sha256:', hashlib.sha256(note).hexdigest()[:16])
sdk = None
for p in pathlib.Path('/kaggle/input').rglob('aicomp_sdk'):
    if (p / 'core/tools').is_dir(): sdk = p.parent; break
print('competition SDK found at:', sdk)


## The 19 checks


In [ ]:
import subprocess, sys, os
env = dict(os.environ)
if sdk: env['JED_SDK_ROOT'] = str(sdk); env['PYTHONPATH'] = str(sdk)
r = subprocess.run([sys.executable, 'verify.py'], cwd=str(work), env=env,
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else '')
print('exit code:', r.returncode)


## What this notebook cannot run

The six model-backed measurements, listed with the command that reproduces each.


In [ ]:
r = subprocess.run([sys.executable, 'verify.py', '--full'], cwd=str(work), env=env,
                   capture_output=True, text=True)
print(r.stdout[-3000:])
